# 64 · Does reciprocal-best pairing inside paralog families change the species results?

The paper's shared space keeps reciprocal-best HCOP pairs (17,449). Inside multi-member orthogroups
it keeps whichever pair scored best on database support. Notebook 62 found 1,051 such pairs. In 59
mouse families the kept member is not the family's main PT paralog (for example SLCO1A2–Slco1a5,
while Slco1a1 and Slco1a6 carry the PT signal). This notebook re-runs notebook 43's primary
species analysis (conservation index and symmetric gene classes) and notebook 49's top strength bins
with the pairs (a) swapped to the main PT paralogs and (b) dropped.

**Summary (numbers from this run).**
- **Guards pass.** The baseline reproduces notebook 43's 8 conservation-index rows (every field to 1e-9),
  its 9,169 symmetric class rows and its class counts exactly. It also reproduces notebook 49's
  top-bin and TOP indices, point and interval.
- **(a) Swap.** 92 pairs change in 90 families: 62 mouse and 34 human members.
  - Primary and external conservation indices change by at most 0.002; the strength-curve top bins by
    at most 0.016.
  - Union class counts go from 196/75/41/76 to 196/72/41/78 (conserved/reversal/mouse-only/
    human-only). Every count is within the rule.
  - **Verdict: unaffected.**
- **(b) Drop.** 1,051 pairs are removed.
  - Indices change by at most 0.007; top bins by at most 0.023.
  - Union counts become 192/71/38/79.
  - One count breaks the rule: S3c − early mouse-only, 13 → 10. None of the 13 genes was dropped. The
    three losses are unchanged genes that become indeterminate once moderation and amplitude scaling are
    re-estimated on the smaller gene set.
  - **Formally affected, in a count the draft does not quote.**
- **Named examples move.** Three of the draft's six named mouse-only examples change class in one
  setting: Comt (swap), Igfbp4 and Ebp (drop) become indeterminate. These genes are not family members,
  so the class of a borderline gene depends on the gene set used for moderation.
  - The human-only examples, the conserved transporter examples, the reversal share (27–28%) and the
    transporter share of conserved genes (32–34 of 192–196) all hold.

Cohort: 2 male control mice and 2 cortex sections from 1 male human donor; reviewed segment labels as
in notebook 43. Descriptive.

## 0 · Protocol (fixed in the workstream C plan before any result of this notebook)

| Item | Rule |
|---|---|
| Baseline | notebook 43's primary analysis rebuilt with its code and constants. It must reproduce `conservation_index.csv` (every field to 1e-9) and `gene_classes_symmetric.csv` (identical) before anything else runs. Notebook 49's published top-bin indices must also be reproduced. |
| Families | notebook 62's orthogroups: HCOP support ≥ 3, class 1:many, many:1 or many:many |
| Main PT paralog | the family member with the highest pooled CP10k among PT-expressed, probe-clean members (notebook 62) |
| (a) Swap | for each family and species side, if the main PT paralog is not a kept member, it replaces the kept member of that side with the lowest pooled CP10k; the number of pairs is unchanged |
| (b) Drop | remove every kept pair whose human or mouse member lies in such a family |
| Probe balance | notebook 38's definition, recomputed for every pair (asserted equal to notebook 38's table for the paper's pairs) |
| Census | genes swapped in that the Census pseudobulks lack are fetched with the same version and filters; control genes reproduce exactly |
| Decision | **unaffected** if every index changes by < 0.03 and every count of the four zonated symmetric classes by < 10% (≤ 2 genes when the baseline count is < 20); otherwise list the draft statements to update |

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
upstream13 = results_root / 'minimal_pt_scfates'
reference43 = results_root / 'pt_revision_classes' / 'reviewed' / 'tables'
reference49 = results_root / 'pt_conservation_strength' / 'tables'
upstream62 = results_root / 'pt_non_one_to_one_orthologs' / 'tables'
external = data_root / 'external'
output = results_root / 'pt_paralog_pairing_check'
tables = output / 'tables'
tables.mkdir(parents=True, exist_ok=True)
cache = output / 'stage_cache'
lake_file = external / 'kpmp_lake2023' / 'lake2023_snRNA_v1.5.h5ad'
census_dir, census_extra = external / 'census_pt_segments', external / 'census_pt_segments_paralogs'
probe38 = results_root / 'pt_literature_deep' / 'probe_counts.csv'
MOUSE_PANEL = external / 'visium_probe_sets' / 'Visium_Mouse_Transcriptome_Probe_Set_v2.0_mm10-2020-A.csv'
MOLECULE_INFO = [data_root / s / 'outs' / 'molecule_info.h5' for s in ('HUK_COR_1', 'HUK_MED_1')]
required = [upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv',
            results_root / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad',
            reference43 / 'conservation_index.csv', reference43 / 'gene_classes_symmetric.csv',
            reference43 / 'class_counts_by_setting_and_contrast.csv', reference49 / 'strength_curves.csv',
            upstream62 / 'gene_inventory_mouse.csv', upstream62 / 'gene_inventory_human.csv',
            external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz',
            external / 'segment_microdissection_sex' / 'GSE212213_TPM_Male_Female.txt.gz',
            *[census_dir / f'{s}_pt_segment_pseudobulk_{k}' for s in ('mouse', 'human') for k in ('counts.csv.gz', 'meta.csv')],
            *[census_extra / f'{s}_pt_segment_pseudobulk_counts.csv.gz' for s in ('mouse', 'human')],
            lake_file, probe38, MOUSE_PANEL, *MOLECULE_INFO]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '64.paralog_pairing.1'
# Notebook 43's constants (unchanged).
SEED, N_BOOT = 43, 1000
MIN_COUNTS, MIN_NUCLEI, REFERENCE_LEVEL = 20, 50, 3.
ALPHA, EFFECT, MARGIN, CONFIRM_ALPHA = .05, .5, .3, .05
# Notebook 49's constants for the strength curve (primary scheme, gene bootstrap only).
SEED49, Q_PUBLISHED, EDGES, MIN_RELIABILITY = 49, .01, (0., .5, .8, .9, .95, .99, 1.), .2
# Decision rule (protocol).
INDEX_TOLERANCE, COUNT_TOLERANCE, SMALL_COUNT, SMALL_TOLERANCE = .03, .10, 20, 2
ZONATED_CLASSES = ['conserved', 'reversal', 'mouse-only', 'human-only']
print('Results folder:', output)

## 1 · Inputs that do not depend on the pairing

Structures, labels, mouse cortical-like S3 (notebook 43, section 2), the Lake cortex pseudobulk over
**all** genes (mapped to each pairing later), microdissection and Census tables. The Census tables are
the published pseudobulks plus the genes swapped in (`census_pt_segments_paralogs`). Its control genes
must equal the published counts.

In [ ]:
import anndata as ad
import h5py
import numpy as np
import pandas as pd
import re
from IPython.display import display
from scipy import sparse
from scipy.spatial import cKDTree

from pseudospace import ortholog_pairing as pairing
from pseudospace.pathway_inputs import rebuild_pt_expression
from pseudospace.replication_rules import csr_group_sums
from pseudospace.repeated_inputs import _read_and_validate_segmentation
from pseudospace.stage_cache import cached_payload, digest
from pseudospace.zonation_amplitude import group_sums, log_ratio
from pseudospace.zonation_classes import moderated_contrast, zonation_calls, confirm, cross_species_class, union_class
from pseudospace.zonation_reliability import paired_gradient, unpaired_gradient, conservation_index, confirm_flat

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 30)
saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r')
pt_obs = saved_pt.obs.copy()
saved_pt.file.close()
pt_obs = pt_obs[pt_obs.broad_tubule_marker_call.astype(str).eq('PT')].copy()
expected = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
names = expected['mouse'] + expected['human']
source_paths = {name: data_root / 'tubule_by_gene' / f'{name}_tubule_by_gene_caleb.h5ad' for name in names}
paper_map = pd.read_csv(upstream13 / 'ortholog_map_used.csv')

human_source = ad.read_h5ad(source_paths[expected['human'][0]], backed='r')
human_var = human_source.var[['gene_ids']].copy()
human_source.file.close()
mouse_source = ad.read_h5ad(source_paths[expected['mouse'][0]], backed='r')
mouse_var = mouse_source.var[['gene_ids']].copy()
mouse_source.file.close()


def lake_all_genes():
    """Notebook 43's Lake cortex PT grouping (donor | disease | sex | segment), summed over every gene."""
    def column(group, name):
        item = group[name]
        if isinstance(item, h5py.Group):
            categories = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in item['categories'][:]])
            return pd.Categorical.from_codes(item['codes'][:], categories).astype(str)
        values = item[:]
        return values.astype(str) if values.dtype.kind in 'SO' else values

    with h5py.File(lake_file, 'r') as handle:
        obs = pd.DataFrame({c: column(handle['obs'], c) for c in ['subclass.l2', 'disease', 'region', 'tissue', 'sex', 'donor_id']})
        var_ids = handle['raw/var/_index'][:].astype(str)
        var_names = np.asarray(column(handle['raw/var'], 'feature_name'))
        cortex = obs.region.eq('C') & obs.tissue.eq('cortex of kidney')
        keep = cortex & obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3'])
        label = (obs.donor_id + '|' + obs.disease + '|' + obs.sex + '|' + obs['subclass.l2'].str.replace('PT-', '', regex=False))[keep]
        groups, codes = np.unique(label.to_numpy(), return_inverse=True)
        raw = handle['raw/X']
        sums, totals, nuclei = csr_group_sums(raw['data'], raw['indices'], raw['indptr'], np.flatnonzero(keep.to_numpy()),
                                              codes, len(groups), len(var_ids), block_rows=5000)
    return {'groups': groups.astype(str), 'nuclei': nuclei, 'totals': totals, 'sums': sums,
            'var_ids': var_ids, 'var_names': var_names.astype(str)}


lake_raw = cached_payload('lake_all_genes', lake_all_genes, root=cache,
                          params={'labels': 'PT-S1,PT-S2,PT-S3', 'region': 'C, cortex of kidney', 'disease': 'all'},
                          inputs={'lake': [lake_file.name, lake_file.stat().st_size]}, code=[lake_all_genes, csr_group_sums])
lake_meta = pd.DataFrame([g.split('|') for g in lake_raw['groups']], columns=['donor', 'disease', 'sex', 'segment'])
lake_meta['nuclei'], lake_meta['total'] = lake_raw['nuclei'], lake_raw['totals']
nuclei_table = lake_meta.pivot_table(index='donor', columns='segment', values='nuclei', aggfunc='sum', fill_value=0)
lake_donors = nuclei_table.index[(nuclei_table.reindex(columns=['S1', 'S2', 'S3'], fill_value=0) >= MIN_NUCLEI).all(axis=1)]
lake_disease = lake_meta.drop_duplicates('donor').set_index('donor').disease
healthy_lake = [d for d in lake_donors if lake_disease[d] == 'normal']

# Census: published pseudobulks plus the swapped-in genes (each species' own symbols).
census_tables = {}
for label in ('mouse', 'human'):
    published = pd.read_csv(census_dir / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    extra = pd.read_csv(census_extra / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    if set(extra.index) != set(published.index):
        raise ValueError(f'Census {label}: the extension has different groups.')
    extra = extra.reindex(published.index)
    if label == 'human':  # published human columns carry the paper's mouse symbols; restore human symbols
        published = published.rename(columns=dict(zip(paper_map.mouse_symbol, paper_map.human_symbol)))
    controls = [c for c in extra.columns if c in published.columns]
    if not controls or not all(np.array_equal(extra[c].to_numpy(), published[c].to_numpy()) for c in controls):
        raise ValueError(f'Census {label}: control genes are not reproduced by the extension.')
    census_tables[label] = pd.concat([published, extra.drop(columns=controls)], axis=1)
    print(f'Census {label}: {published.shape[1]:,} published genes + {extra.shape[1] - len(controls)} swapped-in genes; '
          f'{len(controls)} controls reproduced exactly.')
census_meta = {label: pd.read_csv(census_dir / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0) for label in ('mouse', 'human')}

micro_table = pd.read_csv(external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz', sep='\t')
micro_table = micro_table.drop_duplicates('Gene_symbol').set_index('Gene_symbol')
micro_sex_table = pd.read_csv(external / 'segment_microdissection_sex' / 'GSE212213_TPM_Male_Female.txt.gz', sep='\t')
micro_sex_table = micro_sex_table.drop_duplicates('Gene_symbol').set_index('Gene_symbol')

# Probe counts (notebook 38's definition).
mouse_panel = pd.read_csv(MOUSE_PANEL, comment='#')
mouse_probe_count = mouse_panel[mouse_panel.included.astype(str).str.upper().eq('TRUE')].groupby('gene_id').size()
human_counts = []
for path in MOLECULE_INFO:
    with h5py.File(path, 'r') as handle:
        probes = handle['probes']
        table = pd.DataFrame({'feature_id': probes['feature_id'][:].astype(str), 'included': probes['included_probes'][:].astype(bool)})
    human_counts.append(table[table.included].groupby('feature_id').size())
pd.testing.assert_series_equal(human_counts[0], human_counts[1])
human_probe_count = human_counts[0]


def probe_table_for(pairs):
    return pairing.probe_balance(pairs, mouse_probe_count, human_probe_count, mouse_var.gene_ids, human_var.gene_ids)


published_probe = pd.read_csv(probe38)
check = probe_table_for(paper_map).merge(published_probe[['mouse_symbol', 'human_symbol', 'mouse_probes', 'human_probes', 'probe_balance']],
                                         on=['mouse_symbol', 'human_symbol'], suffixes=('', '_38'))
if len(check) != len(published_probe) or not (check.probe_balance.eq(check.probe_balance_38).all()
                                              and check.mouse_probes.eq(check.mouse_probes_38).all()
                                              and check.human_probes.eq(check.human_probes_38).all()):
    raise ValueError("Probe balance does not reproduce notebook 38's table.")
print(f"Probe balance reproduces notebook 38 for all {len(check):,} paper pairs.")

## 2 · Notebook 43's primary analysis as a function of the pair table

The body follows notebook 43's sections 2–6 line by line: segment pseudobulks, gradients,
references, gradients with noise, conservation index and the symmetric classes. Only the parts that
feed the index, the symmetric classes and notebook 49's strength curve are kept. The rat, female
microdissection and pathway sections do not enter them.

In [ ]:
CONTRASTS = {'S2-S1': ('S2', 'S1'), 'S3c-early': ('S3c', 'early'), 'S3-early': ('S3', 'early')}
D2_CONTRASTS = ('S2-S1', 'S3-early', 'S3c-early')
CLASS_CONTRASTS = ('S2-S1', 'S3c-early', 'S3-early')


def log2_cpm(counts_series, total):
    return np.log2(counts_series / total * 1e6 + 1)


def primary_analysis(orthologs, label):
    """Notebook 43's conservation index and symmetric classes for one pair table."""
    adata = rebuild_pt_expression(pt_obs, source_paths, orthologs, target_sum=1e4)
    measured = adata.var.measured_in_both_inputs.to_numpy()
    gene_space = adata.var_names[measured]
    counts = sparse.csr_matrix(adata.layers['counts'], dtype=float)[:, measured]
    specimen = adata.obs['sample'].astype(str).to_numpy()
    segment = adata.obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
    library = np.asarray(counts.sum(axis=1)).ravel()

    # Cortical-like mouse S3 (notebook 43, section 2): depth to the three nearest glomeruli.
    def mouse_depth():
        pass1 = ad.read_h5ad(results_root / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad', backed='r')
        anchors = pass1.obs[['sample', 'coarse_class', 'feature_index', 'x_centroid', 'y_centroid']].copy()
        pass1.file.close()
        values = np.full(len(specimen), np.nan)
        for name in expected['mouse']:
            native = ad.read_h5ad(source_paths[name], backed='r')
            source_obs = native.obs[['feature_index', 'x_centroid', 'y_centroid']].copy()
            native.file.close()
            mask = specimen == name
            structures = source_obs.loc[adata.obs_names[mask].str[len(name) + 1:]]
            _read_and_validate_segmentation(data_root, name, structures)
            glomeruli = anchors[anchors['sample'].astype(str).eq(name) & anchors.coarse_class.astype(str).eq('Glomerulus')]
            _read_and_validate_segmentation(data_root, name, glomeruli)
            distance, _ = cKDTree(glomeruli[['x_centroid', 'y_centroid']].to_numpy(float)).query(
                structures[['x_centroid', 'y_centroid']].to_numpy(float), k=3)
            values[mask] = distance.mean(axis=1)
        return {'depth': values}

    depth = cached_payload('mouse_depth', mouse_depth, root=cache, inputs={'obs': list(adata.obs_names)},
                           code=digest(NOTEBOOK_LOGIC_VERSION))['depth']
    cortical_s3 = np.zeros(len(specimen), bool)
    for name in expected['mouse']:
        limit = np.quantile(depth[(specimen == name) & (segment == 'S1')], .95)
        cortical_s3 |= (specimen == name) & (segment == 'S3') & (depth <= limit)
    segment_region = np.where((segment == 'S3') & np.isin(specimen, expected['mouse']) & ~cortical_s3, 'S3deep', segment)

    names_, sums = group_sums(counts, np.char.add(np.char.add(specimen.astype(str), '|'), segment_region.astype(str)))
    frame = pd.DataFrame(sums, index=names_, columns=gene_space)
    ours_counts = {}
    for name in np.unique(specimen):
        rows = lambda labels: [f'{name}|{s}' for s in labels if f'{name}|{s}' in frame.index]
        ours_counts[(name, 'S1')] = frame.loc[rows(['S1'])].sum()
        ours_counts[(name, 'S2')] = frame.loc[rows(['S2'])].sum()
        ours_counts[(name, 'S3')] = frame.loc[rows(['S3', 'S3deep'])].sum()
        ours_counts[(name, 'S3c')] = frame.loc[rows(['S3'])].sum()
        ours_counts[(name, 'early')] = ours_counts[(name, 'S1')] + ours_counts[(name, 'S2')]
    ours_counts = pd.DataFrame(ours_counts)
    ours_library = pd.Series({key: float(library[(specimen == key[0]) & np.isin(
        segment_region, {'S1': ['S1'], 'S2': ['S2'], 'S3': ['S3', 'S3deep'], 'S3c': ['S3'], 'early': ['S1', 'S2']}[key[1]])].sum())
        for key in ours_counts.columns})
    ours = pd.DataFrame({(contrast, name): log_ratio(ours_counts[(name, late)], ours_library[(name, late)],
                                                     ours_counts[(name, early)], ours_library[(name, early)])
                         for contrast, (late, early) in CONTRASTS.items() for name in names}, index=ours_counts.index)

    def count_filter(specimens, parts):
        return ours_counts[[(n, p) for n in specimens for p in parts]].min(axis=1).ge(MIN_COUNTS)

    # References (notebook 43, section 4).
    per_mouse = {}
    for column in micro_table.columns:
        match = re.fullmatch(r'(m\d+)_PTS([123])_[a-z]', column)
        if match:
            per_mouse.setdefault((f'S{match.group(2)}', match.group(1)), []).append(column)
    micro_male = pd.DataFrame({key: np.log2(micro_table[cols].apply(pd.to_numeric, errors='coerce').clip(lower=0) + 1).mean(axis=1)
                               for key, cols in per_mouse.items()}).reindex(gene_space)

    def census_segments(label_, segment_map):
        table = census_tables[label_]
        if label_ == 'human':
            table = table.rename(columns=dict(zip(orthologs.human_symbol, orthologs.mouse_symbol)))
            table = table.loc[:, table.columns.isin(gene_space)]
            if table.columns.duplicated().any():
                raise ValueError('Census human columns collide after renaming to mouse symbols.')
        meta = census_meta[label_]
        meta = meta[~meta.development_stage.str.contains('embryo|E16|Theiler', case=False)].copy()
        meta['segment'] = meta.cell_type.map(segment_map)
        meta = meta[meta.segment.notna() & meta.n_cells.ge(MIN_NUCLEI)]
        needed = set(segment_map.values())
        complete = meta.groupby('donor_id').segment.apply(lambda s: needed.issubset(set(s)))
        meta = meta[meta.donor_id.isin(complete[complete].index)]
        counts_out, totals, info = {}, {}, {}
        for (donor, sex), block in meta.groupby(['donor_id', 'sex']):
            for seg_name in needed:
                index = block.index[block.segment.eq(seg_name)]
                counts_out[(donor, seg_name)] = table.loc[index].sum()
                totals[(donor, seg_name)] = meta.loc[index, 'total_counts_all_genes'].sum()
            info[donor] = sex
        return pd.DataFrame(counts_out).reindex(gene_space).fillna(0), pd.Series(totals), pd.Series(info)

    sn_counts, sn_totals, sn_sex = census_segments('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                                                             'epithelial cell of proximal tubule segment 2': 'S2',
                                                             'epithelial cell of proximal tubule segment 3': 'S3'})
    census_h_counts, census_h_totals, census_h_sex = census_segments('human', {
        'kidney proximal convoluted tubule epithelial cell': 'early', 'epithelial cell of proximal tubule segment 3': 'S3'})

    by_id = dict(zip(orthologs.human_symbol.map(human_var.gene_ids), orthologs.mouse_symbol))
    by_symbol = dict(zip(orthologs.human_symbol, orthologs.mouse_symbol))
    target = np.array([by_id.get(i, by_symbol.get(s, '')) for i, s in zip(lake_raw['var_ids'], lake_raw['var_names'])])
    mapped = target != ''
    lake_sums = (pd.DataFrame(lake_raw['sums'][:, mapped], columns=target[mapped]).T.groupby(level=0).sum().T
                 .reindex(columns=gene_space, fill_value=0.))
    lake_counts, lake_totals = {}, {}
    for donor in lake_donors:
        for seg_name in ('S1', 'S2', 'S3'):
            rows = lake_meta.index[lake_meta.donor.eq(donor) & lake_meta.segment.eq(seg_name)]
            lake_counts[(donor, seg_name)] = lake_sums.loc[rows].sum()
            lake_totals[(donor, seg_name)] = float(lake_meta.loc[rows, 'total'].sum())
    lake_counts, lake_totals = pd.DataFrame(lake_counts), pd.Series(lake_totals)

    def donor_contrasts(count_frame, totals, donors, contrast):
        late, early = CONTRASTS[contrast]
        late = 'S3' if late == 'S3c' else late
        rows_contrast, rows_level = {}, {}
        for donor in donors:
            def level(parts):
                pooled = sum(count_frame[(donor, p)] for p in parts)
                return log2_cpm(pooled, sum(totals[(donor, p)] for p in parts))
            late_level = level([late])
            early_level = level(['S1', 'S2'] if early == 'early' else [early])
            rows_contrast[donor] = late_level - early_level
            rows_level[donor] = np.maximum(late_level, early_level)
        return pd.DataFrame(rows_contrast).T, pd.DataFrame(rows_level).T

    # Gradients with noise (notebook 43, section 5).
    def ours_gradient(contrast, specimens):
        late, early = CONTRASTS[contrast]
        expressed_ = set(gene_space[count_filter(specimens, [late, early]).to_numpy()])
        return paired_gradient(ours[[(contrast, n) for n in specimens]].T.reset_index(drop=True)), expressed_

    def donor_gradient(count_frame, totals, donors, contrast):
        per_donor, level = donor_contrasts(count_frame, totals, donors, contrast)
        keep = level.mean().ge(REFERENCE_LEVEL) & per_donor.notna().all()
        return paired_gradient(per_donor.loc[:, keep]), set(keep.index[keep.to_numpy()])

    def census_human_gradient():
        donors = census_h_sex.index
        late = pd.DataFrame({d: log2_cpm(census_h_counts[(d, 'S3')], census_h_totals[(d, 'S3')]) for d in donors}).T
        early = pd.DataFrame({d: log2_cpm(census_h_counts[(d, 'early')], census_h_totals[(d, 'early')]) for d in donors}).T
        keep = np.maximum(late.mean(), early.mean()).ge(REFERENCE_LEVEL) & (late - early).notna().all()
        return paired_gradient((late - early).loc[:, keep]), set(keep.index[keep.to_numpy()])

    micro_male_segments = {s: micro_male[[c for c in micro_male.columns if c[0] == s]].T.reset_index(drop=True) for s in ('S1', 'S2', 'S3')}

    def segment_replicates_gradient(by_segment, contrast):
        late, early = CONTRASTS[contrast]
        late = 'S3' if late == 'S3c' else late
        if early == 'early':
            parts = [by_segment['S1'], by_segment['S2']]
            mean = by_segment[late].mean() - (parts[0].mean() + parts[1].mean()) / 2
            noise = by_segment[late].var(ddof=1) / len(by_segment[late]) + sum(p.var(ddof=1) / len(p) for p in parts) / 4
            level = pd.concat([by_segment[late].mean(), (parts[0].mean() + parts[1].mean()) / 2], axis=1).max(axis=1)
        else:
            mean, noise = unpaired_gradient(by_segment[late], by_segment[early])
            level = pd.concat([by_segment[late].mean(), by_segment[early].mean()], axis=1).max(axis=1)
        keep = level.ge(REFERENCE_LEVEL) & mean.notna()
        return (mean, noise), set(keep.index[keep.to_numpy()])

    male_sn, female_sn = sn_sex.index[sn_sex.eq('male')], sn_sex.index[sn_sex.eq('female')]

    def datasets_for(contrast):
        human_contrast = 'S3-early' if contrast == 'S3c-early' else contrast
        out = {('human', 'ours'): ours_gradient(contrast, expected['human']),
               ('mouse', 'ours'): ours_gradient(contrast, expected['mouse']),
               ('human', 'Lake healthy'): donor_gradient(lake_counts, lake_totals, healthy_lake, human_contrast),
               ('mouse', 'snRNA male'): donor_gradient(sn_counts, sn_totals, male_sn, human_contrast),
               ('mouse', 'micro male GSE150338'): segment_replicates_gradient(micro_male_segments, human_contrast)}
        if contrast != 'S2-S1':
            out[('human', 'Census human')] = census_human_gradient()
        return out

    datasets = {contrast: datasets_for(contrast) for contrast in D2_CONTRASTS}
    index_rows = []
    for contrast, data in datasets.items():
        h, m, l, s = data[('human', 'ours')], data[('mouse', 'ours')], data[('human', 'Lake healthy')], data[('mouse', 'snRNA male')]
        genes = pd.Index(sorted(h[1] & m[1] & l[1] & s[1]))
        index_rows.append({'contrast': contrast, 'version': 'primary: ours human vs ours mouse',
                           **conservation_index(h[0], m[0], l[0], s[0], genes, n_boot=N_BOOT, seed=SEED)})
        index_rows.append({'contrast': contrast, 'version': 'external: Lake vs mouse snRNA male',
                           **conservation_index(l[0], s[0], h[0], m[0], genes, n_boot=N_BOOT, seed=SEED)})
        micro = data[('mouse', 'micro male GSE150338')]
        if ('human', 'Census human') in data:
            c = data[('human', 'Census human')]
            genes_alt = pd.Index(sorted(l[1] & s[1] & c[1] & micro[1]))
            index_rows.append({'contrast': contrast, 'version': 'external, alternative ceilings: Lake vs snRNA male (Census human, microdissection)',
                               **conservation_index(l[0], s[0], c[0], micro[0], genes_alt, n_boot=N_BOOT, seed=SEED)})
    conservation = pd.DataFrame(index_rows)

    # Symmetric classes (notebook 43, section 6; setting 'symmetric (A3b + A8)').
    def species_level(specimens, parts):
        return pd.concat([log2_cpm(ours_counts[(n, p)], ours_library[(n, p)]) for n in specimens for p in parts], axis=1).mean(axis=1)

    def reference_table(count_frame, totals, donors, contrast):
        per_donor, level = donor_contrasts(count_frame, totals, donors, contrast)
        per_donor = per_donor.loc[:, level.mean().ge(REFERENCE_LEVEL) & per_donor.notna().all()]
        return moderated_contrast(per_donor, level.mean())

    tables_ours, tables_ref = {}, {}
    for contrast in CLASS_CONTRASTS:
        late, early = CONTRASTS[contrast]
        both = gene_space[(count_filter(expected['mouse'], [late, early]) & count_filter(expected['human'], [late, early])).to_numpy()]
        for species in ('mouse', 'human'):
            replicates = ours.loc[both, [(contrast, n) for n in expected[species]]].T.reset_index(drop=True)
            tables_ours[(contrast, species)] = moderated_contrast(replicates, species_level(expected[species], [late, early]))
        tables_ref[(contrast, 'mouse male')] = reference_table(sn_counts, sn_totals, male_sn, contrast)
        tables_ref[(contrast, 'human')] = reference_table(lake_counts, lake_totals, healthy_lake, contrast)

    def noise_corrected_amplitude(contrast):
        genes_ = tables_ours[(contrast, 'mouse')].index

        def signal(specimens):
            reps = ours.loc[genes_, [(contrast, n) for n in specimens]].to_numpy()
            return reps.mean(axis=1).var(ddof=1) - reps.var(axis=1, ddof=1).mean() / reps.shape[1]
        return float(np.sqrt(max(signal(expected['human']), 0) / signal(expected['mouse'])))

    amplitude = {c: noise_corrected_amplitude(c) for c in CLASS_CONTRASTS}
    frame_ = {}
    for contrast in CLASS_CONTRASTS:
        scale = amplitude[contrast]
        mouse_calls, _ = zonation_calls(tables_ours[(contrast, 'mouse')], alpha=ALPHA, effect=EFFECT, margin=MARGIN)
        mouse_calls = confirm(mouse_calls, tables_ref[(contrast, 'mouse male')], alpha=CONFIRM_ALPHA)
        human_calls, _ = zonation_calls(tables_ours[(contrast, 'human')], alpha=ALPHA, effect=EFFECT * scale, margin=MARGIN * scale)
        human_calls = confirm(human_calls, tables_ref[(contrast, 'human')], alpha=CONFIRM_ALPHA)
        classes = pd.Series([cross_species_class(a, b) for a, b in zip(mouse_calls, human_calls)], index=mouse_calls.index)
        classes = confirm_flat(classes, mouse_calls, human_calls, tables_ref[(contrast, 'mouse male')], tables_ref[(contrast, 'human')],
                               alpha=CONFIRM_ALPHA, effect=EFFECT)
        frame_[contrast], frame_[contrast + ' mouse call'], frame_[contrast + ' human call'] = classes, mouse_calls, human_calls
    symmetric = pd.DataFrame(frame_)
    symmetric['union S2-S1 + S3c-early'] = union_class(symmetric[['S2-S1', 'S3c-early']])
    probe = probe_table_for(orthologs).drop_duplicates('mouse_symbol').set_index('mouse_symbol')
    balanced = set(probe.index[probe.probe_balance.eq('balanced')])
    symmetric = symmetric.assign(probe_balanced=symmetric.index.isin(balanced))
    ORDER = ['conserved', 'reversal', 'mouse-only', 'human-only', 'indeterminate', 'neither']
    rows = []
    for contrast in (*CLASS_CONTRASTS, 'union S2-S1 + S3c-early'):
        counts_ = symmetric.loc[symmetric.probe_balanced, contrast].value_counts().reindex(ORDER).fillna(0).astype(int)
        rows.append({'setting': label, 'contrast': contrast, **counts_.to_dict(),
                     'mouse-only ÷ human-only': counts_['mouse-only'] / max(counts_['human-only'], 1)})
    index_genes = {c: pd.Index(sorted(d[('human', 'ours')][1] & d[('mouse', 'ours')][1] & d[('human', 'Lake healthy')][1]
                                      & d[('mouse', 'snRNA male')][1])) for c, d in datasets.items()}
    return {'conservation': conservation.assign(setting=label), 'symmetric': symmetric, 'class_counts': pd.DataFrame(rows),
            'datasets': datasets, 'index_genes': index_genes, 'gene_space': gene_space, 'amplitude': amplitude,
            'lake_counts': lake_counts, 'lake_totals': lake_totals, 'sn_counts': sn_counts, 'sn_totals': sn_totals,
            'male_sn': male_sn, 'donor_contrasts': donor_contrasts}

## 3 · Guard: the baseline reproduces notebook 43 exactly

Validation cell (not cached).

In [ ]:
baseline = primary_analysis(paper_map, 'baseline (paper map)')
published_index = pd.read_csv(reference43 / 'conservation_index.csv')
FIELDS = ('index', 'ci_low', 'ci_high', 'cross_corrected', 'ceiling_human', 'ceiling_mouse', 'genes')
merged = published_index.merge(baseline['conservation'], on=['contrast', 'version'], suffixes=('_43', ''))
if len(merged) != len(published_index):
    raise ValueError('Conservation-index rows do not match notebook 43.')
for field in FIELDS:
    if not np.allclose(merged[field].astype(float), merged[field + '_43'].astype(float), rtol=0, atol=1e-9):
        raise ValueError(f'Notebook 43 not reproduced: {field}')
published_classes = pd.read_csv(reference43 / 'gene_classes_symmetric.csv', index_col=0)
mine = baseline['symmetric'].reindex(columns=published_classes.columns)
mine.index.name = published_classes.index.name
pd.testing.assert_frame_equal(mine.sort_index().astype(str), published_classes.sort_index().astype(str))
published_counts = pd.read_csv(reference43 / 'class_counts_by_setting_and_contrast.csv')
published_counts = published_counts[published_counts.setting.eq('symmetric (A3b + A8)')].set_index('contrast')
assert (baseline['class_counts'].set_index('contrast')[ZONATED_CLASSES] == published_counts[ZONATED_CLASSES]).all().all()
print(f'Notebook 43 reproduced exactly: {len(merged)} conservation-index rows, {len(published_classes):,} symmetric class rows, '
      'class counts identical.')

## 4 · Notebook 49's strength curve, top bins (primary scheme, gene bootstrap)

Strength is cross-fitted on external donor folds: the maximum of Lake and snRNA percentile ranks
from fold A, with evaluation on fold B, then swapped. Only the top bin (99–100%) and TOP (≥ 95%)
indices are computed. The baseline must reproduce notebook 49's published values (point and gene
interval).

In [ ]:
from pseudospace.conservation_strength import (binned_statistics, bin_labels, donor_folds, gene_draws, percentile_interval,
                                               percentile_rank, strength_bins, strength_score, STATS)

BINS = bin_labels(EDGES)
SUBSETS = {**{lab: [i] for i, lab in enumerate(BINS)}, 'BOTTOM (<80%)': [0, 1], 'TOP (≥95%)': [4, 5], 'all genes': list(range(len(BINS)))}
SUBSET_NAMES = list(SUBSETS)
TOP_SUBSETS = (BINS[-1], 'TOP (≥95%)')


def strength_top(result):
    lake_fold = pd.Series(donor_folds(healthy_lake), index=healthy_lake)
    sn_fold = pd.Series(donor_folds(list(result['male_sn'])), index=result['male_sn'])
    rows = []
    for contrast, genes in result['index_genes'].items():
        rc = 'S3-early' if contrast == 'S3c-early' else contrast
        d = result['datasets'][contrast]
        lake_pd = result['donor_contrasts'](result['lake_counts'], result['lake_totals'], healthy_lake, rc)[0][genes]
        sn_pd = result['donor_contrasts'](result['sn_counts'], result['sn_totals'], result['male_sn'], rc)[0][genes]

        def arrays(gradient):
            mean, noise = gradient
            return mean.reindex(genes).to_numpy(float), noise.reindex(genes).to_numpy(float)

        def fold_gradient(per_donor, folds, fold):
            mean, noise = paired_gradient(per_donor.loc[folds.index[folds.eq(fold)]])
            return mean.to_numpy(float), noise.to_numpy(float)

        directions = []
        for f in (0, 1):
            score = strength_score([percentile_rank(fold_gradient(lake_pd, lake_fold, f)[0]),
                                    percentile_rank(fold_gradient(sn_pd, sn_fold, f)[0])], 'max')
            data = {'human': arrays(d[('human', 'ours')][0]), 'mouse': arrays(d[('mouse', 'ours')][0]),
                    'human_ref': fold_gradient(lake_pd, lake_fold, 1 - f), 'mouse_ref': fold_gradient(sn_pd, sn_fold, 1 - f)}
            directions.append((data, strength_bins(score, EDGES)))
        n = len(genes)
        point = binned_statistics(directions, np.arange(n), SUBSETS, q=Q_PUBLISHED, min_reliability=MIN_RELIABILITY)
        draws = np.stack([binned_statistics(directions, idx, SUBSETS, q=Q_PUBLISHED, min_reliability=MIN_RELIABILITY)
                          for idx in gene_draws(n, N_BOOT, SEED49)])
        low, high, _ = percentile_interval(draws)
        for subset in TOP_SUBSETS:
            i, j = SUBSET_NAMES.index(subset), STATS.index('index')
            rows.append({'contrast': contrast, 'subset': subset, 'index': point[i, j], 'ci_low': low[i, j], 'ci_high': high[i, j],
                         'genes': point[i, STATS.index('genes')]})
    return pd.DataFrame(rows)


def cached_strength(result, label, stage):
    payload = cached_payload(f'strength_{stage}', lambda: {'table': strength_top(result).to_numpy(dtype=object).astype(str)},
                             root=cache, params={'edges': EDGES, 'q': Q_PUBLISHED, 'n_boot': N_BOOT, 'seed': SEED49},
                             inputs={'genes': {c: list(g) for c, g in result['index_genes'].items()},
                                     'sn': digest(result['sn_counts'].sort_index(axis=1)), 'lake': digest(result['lake_counts']),
                                     'ours': digest({c: [result['datasets'][c][k][0][0] for k in (('human', 'ours'), ('mouse', 'ours'))]
                                                     for c in result['datasets']})},
                             code=[strength_top, binned_statistics, NOTEBOOK_LOGIC_VERSION])
    table = pd.DataFrame(payload['table'], columns=['contrast', 'subset', 'index', 'ci_low', 'ci_high', 'genes'])
    return table.astype({'index': float, 'ci_low': float, 'ci_high': float, 'genes': float}).assign(setting=label)


strength = {'baseline (paper map)': cached_strength(baseline, 'baseline (paper map)', 'baseline')}
published49 = pd.read_csv(reference49 / 'strength_curves.csv')
published49 = published49[published49.scheme.eq('max rank (primary)') & published49.q.eq(Q_PUBLISHED)
                          & published49.bootstrap.eq('gene') & published49.statistic.eq('index') & published49.subset.isin(TOP_SUBSETS)]
check49 = strength['baseline (paper map)'].merge(published49, on=['contrast', 'subset'])
if len(check49) != 6 or not (np.allclose(check49['index'], check49['value'], atol=1e-9)
                             and np.allclose(check49['ci_low_x'], check49['ci_low_y'], atol=1e-9)
                             and np.allclose(check49['ci_high_x'], check49['ci_high_y'], atol=1e-9)):
    raise ValueError("Notebook 49's published top-bin indices are not reproduced.")
print("Notebook 49's top-bin and TOP indices reproduced exactly (point and gene-bootstrap interval).")

## 5 · The two alternative pairings

In [ ]:
mouse62 = pd.read_csv(upstream62 / 'gene_inventory_mouse.csv', index_col=0)
human62 = pd.read_csv(upstream62 / 'gene_inventory_human.csv', index_col=0)
swap_map, swap_log = pairing.swap_pairs(paper_map, mouse62, human62)
drop_map, dropped_pairs = pairing.drop_pairs(paper_map, mouse62, human62)
print(f'Swap: {len(swap_log)} pairs changed in {swap_log.component.nunique()} families '
      f'({int(swap_log["mouse changed"].sum())} mouse members, {int(swap_log["human changed"].sum())} human members); '
      f'{len(swap_map):,} pairs.')
print(f'Drop: {len(dropped_pairs):,} pairs removed; {len(drop_map):,} pairs remain.')
swap_log.to_csv(tables / 'swap_log.csv', index=False)
dropped_pairs.to_csv(tables / 'dropped_pairs.csv', index=False)
swapped_in = set(swap_log['new pair'].str.split('–').str[1]) - set(paper_map.mouse_symbol)
missing_census = sorted(swapped_in - set(census_tables['mouse'].columns))
print('Swapped-in mouse genes absent from Census (not in the Census gene set):', missing_census)

results = {'baseline (paper map)': baseline,
           '(a) swap to main PT paralog': primary_analysis(swap_map, '(a) swap to main PT paralog'),
           '(b) drop family pairs': primary_analysis(drop_map, '(b) drop family pairs')}
strength['(a) swap to main PT paralog'] = cached_strength(results['(a) swap to main PT paralog'], '(a) swap to main PT paralog', 'swap')
strength['(b) drop family pairs'] = cached_strength(results['(b) drop family pairs'], '(b) drop family pairs', 'drop')

## 6 · Indices, class counts and the decision

In [ ]:
index_table = pd.concat([r['conservation'] for r in results.values()], ignore_index=True)
index_wide = index_table.pivot_table(index=['contrast', 'version'], columns='setting', values='index', sort=False)
for setting in list(results)[1:]:
    index_wide[f'Δ {setting}'] = index_wide[setting] - index_wide['baseline (paper map)']
ci_wide = index_table.assign(ci=lambda d: d.apply(lambda r: f'{r["index"]:.3f} ({r.ci_low:.3f}–{r.ci_high:.3f}); n={int(r.genes)}', axis=1)) \
    .pivot_table(index=['contrast', 'version'], columns='setting', values='ci', aggfunc='first', sort=False)
display(ci_wide)
display(index_wide.round(4))
index_table.to_csv(tables / 'conservation_index_by_setting.csv', index=False)

strength_table = pd.concat(strength.values(), ignore_index=True)
strength_wide = strength_table.pivot_table(index=['contrast', 'subset'], columns='setting', values='index', sort=False)
for setting in list(results)[1:]:
    strength_wide[f'Δ {setting}'] = strength_wide[setting] - strength_wide['baseline (paper map)']
display(strength_table.assign(ci=lambda d: d['index'].round(3).astype(str) + ' (' + d.ci_low.round(3).astype(str) + '–'
                              + d.ci_high.round(3).astype(str) + ')')
        .pivot_table(index=['contrast', 'subset'], columns='setting', values='ci', aggfunc='first', sort=False))
strength_table.to_csv(tables / 'strength_top_bins_by_setting.csv', index=False)

counts_table = pd.concat([r['class_counts'] for r in results.values()], ignore_index=True)
counts_wide = counts_table.set_index(['setting', 'contrast'])[ZONATED_CLASSES + ['indeterminate', 'neither', 'mouse-only ÷ human-only']]
display(counts_wide.round(2))
counts_table.to_csv(tables / 'class_counts_by_setting.csv', index=False)
shares = counts_table.assign(classified=counts_table[['conserved', 'reversal', 'mouse-only', 'human-only', 'indeterminate', 'neither']].sum(axis=1))
for c in ZONATED_CLASSES:
    shares[c + ' share'] = shares[c] / shares.classified
display(shares.set_index(['setting', 'contrast'])[[c + ' share' for c in ZONATED_CLASSES] + ['classified']].round(4))

base_counts = counts_table[counts_table.setting.eq('baseline (paper map)')].set_index('contrast')[ZONATED_CLASSES]
decision_rows = []
for setting in list(results)[1:]:
    other = counts_table[counts_table.setting.eq(setting)].set_index('contrast')[ZONATED_CLASSES]
    diff = (other - base_counts)
    relative = diff / base_counts.replace(0, np.nan)
    count_ok = np.where(base_counts >= SMALL_COUNT, relative.abs() < COUNT_TOLERANCE, diff.abs() <= SMALL_TOLERANCE)
    index_delta = index_wide[f'Δ {setting}'][index_wide.index.get_level_values('version').str.startswith(('primary', 'external: '))]
    strength_delta = strength_wide[f'Δ {setting}']
    failures = ([f'{c} · {v}: Δ {d:+.3f}' for (c, v), d in index_delta.items() if abs(d) >= INDEX_TOLERANCE]
                + [f'strength {c} · {s}: Δ {d:+.3f}' for (c, s), d in strength_delta.items() if abs(d) >= INDEX_TOLERANCE]
                + [f'{contrast} · {klass}: {int(base_counts.loc[contrast, klass])} → {int(other.loc[contrast, klass])}'
                   for i, contrast in enumerate(base_counts.index) for j, klass in enumerate(ZONATED_CLASSES) if not count_ok[i, j]])
    decision_rows.append({'setting': setting, 'max |Δ index| (primary, external)': float(index_delta.abs().max()),
                          'max |Δ index| (strength top bins)': float(strength_delta.abs().max()),
                          'max relative count change': float(relative.abs().max().max()),
                          'unaffected': not failures, 'failures': '; '.join(failures)})
decision = pd.DataFrame(decision_rows)
decision.to_csv(tables / 'decision.csv', index=False)
display(decision)

## 7 · Which genes change class?

**Swap:** for every changed pair, the class of the old pair (by its mouse symbol, baseline) against
the class of the new pair (new mouse symbol, swap setting), per contrast. **Both settings:** genes
present in both maps whose class changes. Moderated variances, amplitude scaling and reference
filters are estimated over the whole gene set, so a few unchanged pairs can move.

In [ ]:
base_sym = results['baseline (paper map)']['symmetric']
COLUMNS = ['S2-S1', 'S3c-early', 'S3-early', 'union S2-S1 + S3c-early']
swap_sym = results['(a) swap to main PT paralog']['symmetric']
pair_rows = []
for r in swap_log.itertuples():
    old_h, old_m = r._2.split('–', 1)
    new_h, new_m = r._3.split('–', 1)
    row = {'component': r.component, 'kept pair': r._2, 'new pair': r._3}
    for c in COLUMNS:
        row[f'{c} before'] = base_sym[c].get(old_m, 'not classified')
        row[f'{c} after'] = swap_sym[c].get(new_m, 'not classified')
    row['probe-balanced before / after'] = f"{base_sym.probe_balanced.get(old_m, False)} / {swap_sym.probe_balanced.get(new_m, False)}"
    pair_rows.append(row)
swap_classes = pd.DataFrame(pair_rows)
changed_zonated = swap_classes[np.logical_or.reduce([swap_classes[f'{c} before'].ne(swap_classes[f'{c} after'])
                                                     & (swap_classes[f'{c} before'].isin(ZONATED_CLASSES)
                                                        | swap_classes[f'{c} after'].isin(ZONATED_CLASSES)) for c in COLUMNS])]
swap_classes.to_csv(tables / 'swap_pair_classes.csv', index=False)
print(f'{len(changed_zonated)} of {len(swap_classes)} swapped pairs change into or out of a zonated class in at least one contrast:')
display(changed_zonated[['kept pair', 'new pair'] + [f'{c} {w}' for c in COLUMNS for w in ('before', 'after')]])

retained_rows = []
for setting in list(results)[1:]:
    other = results[setting]['symmetric']
    common = base_sym.index.intersection(other.index)
    for c in COLUMNS:
        moved = common[(base_sym.loc[common, c].astype(str) != other.loc[common, c].astype(str)).to_numpy()]
        for g in moved:
            retained_rows.append({'setting': setting, 'contrast': c, 'gene': g, 'before': base_sym.loc[g, c], 'after': other.loc[g, c],
                                  'probe-balanced': bool(base_sym.loc[g, 'probe_balanced'])})
retained = pd.DataFrame(retained_rows, columns=['setting', 'contrast', 'gene', 'before', 'after', 'probe-balanced'])
retained.to_csv(tables / 'class_changes_of_unchanged_pairs.csv', index=False)
print('Unchanged pairs whose class moves:')
display(retained.groupby(['setting', 'contrast', 'before', 'after']).size().rename('genes').reset_index())
dropped_classes = base_sym.reindex(dropped_pairs.mouse_symbol)
print('Baseline classes of the dropped pairs (probe-balanced genes):')
display(pd.DataFrame({c: dropped_classes.loc[dropped_classes.probe_balanced.eq(True), c].value_counts() for c in COLUMNS}).fillna(0).astype(int))

## 7b · Statements in `docs/paper/draft.md` that rest on these classes

The draft (R3; Figure 3) quotes union-class counts, the S2 − S1 counts, the species-only ratio, the
reversal share, the transporter share of conserved genes and named example genes. Each is recomputed
under both settings. The example genes are listed with their union class (S2 − S1 for Slc9a3, as
quoted).

In [ ]:
QUOTED_GENES = {'mouse-only examples (R3)': ['Fah', 'Comt', 'Maoa', 'Igfbp4', 'Ebp', 'Hsd17b4'],
                'human-only examples (R3)': ['Sel1l3', 'Ace2', 'Ahcyl1', 'Hipk2', 'Wwtr1', 'Prkaa2', 'Taf4'],
                'conserved transporter examples (R3)': ['Slc5a2', 'Slc5a1', 'Slc5a8', 'Slc7a13', 'Slc6a18', 'Slc22a7', 'Slc22a12',
                                                        'Slc34a3', 'Agt', 'Ace', 'Enpep']}
UNION = 'union S2-S1 + S3c-early'
statement_rows = []
for setting, result in results.items():
    sym = result['symmetric']
    balanced_sym = sym[sym.probe_balanced]
    union = balanced_sym[UNION].value_counts()
    zonated_both = int(union.get('conserved', 0) + union.get('reversal', 0))
    transporter = balanced_sym.index.str.match(r'^(Slc|Abc|Aqp)')
    s21 = balanced_sym['S2-S1'].value_counts()
    statement_rows.append({
        'setting': setting,
        'union conserved / reversal / mouse-only / human-only': '/'.join(str(int(union.get(c, 0))) for c in ZONATED_CLASSES),
        'S2−S1 tested; conserved / reversal / mouse-only / human-only': f"{int(balanced_sym['S2-S1'].notna().sum())}; "
            + '/'.join(str(int(s21.get(c, 0))) for c in ZONATED_CLASSES),
        'tested in S2−S1 or S3c−early': int(balanced_sym[['S2-S1', 'S3c-early']].notna().any(axis=1).sum()),
        'reversal share of genes zonated in both': f"{int(union.get('reversal', 0))}/{zonated_both} = {union.get('reversal', 0) / zonated_both:.0%}",
        'transporters among conserved': f"{int((balanced_sym[UNION].eq('conserved') & transporter).sum())}/{int(union.get('conserved', 0))}",
        'mouse-only : human-only': f"{int(union.get('mouse-only', 0))} : {int(union.get('human-only', 0))}"})
statements = pd.DataFrame(statement_rows).set_index('setting').T
display(statements)
gene_rows = []
for group, genes_ in QUOTED_GENES.items():
    for g in genes_:
        gene_rows.append({'group': group, 'gene': g, **{setting: (r['symmetric'][UNION].get(g, 'not classified')) for setting, r in results.items()}})
gene_rows.append({'group': 'human-only example, S2 − S1 (R3)', 'gene': 'Slc9a3',
                  **{setting: r['symmetric']['S2-S1'].get('Slc9a3', 'not classified') for setting, r in results.items()}})
quoted = pd.DataFrame(gene_rows)
quoted['changes'] = quoted[list(results)].nunique(axis=1).gt(1)
display(quoted)
statements.to_csv(tables / 'draft_statements_by_setting.csv')
quoted.to_csv(tables / 'draft_example_genes_by_setting.csv', index=False)

## 8 · Run record

In [ ]:
manifest = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION,
            'baseline_reproduced': {'notebook 43 conservation index rows': int(len(merged)),
                                    'notebook 43 symmetric class rows': int(len(published_classes)),
                                    'notebook 49 top-bin rows': int(len(check49))},
            'swap': {'pairs changed': int(len(swap_log)), 'families': int(swap_log.component.nunique())},
            'drop': {'pairs removed': int(len(dropped_pairs))},
            'decision': decision.to_dict(orient='records'),
            'conservation_index': index_table.round(4).to_dict(orient='records'),
            'strength_top_bins': strength_table.round(4).to_dict(orient='records'),
            'class_counts': counts_table.round(4).to_dict(orient='records')}
(output / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print('Wrote', sorted(p.name for p in tables.iterdir()))

## 9 · Decision and what to change in the draft

- **Species results.** The conservation index, the strength-curve top bins and the class counts
  quoted in R3 and Figure 3 do not depend on which paralog the reciprocal-best map kept:
  - every index moves by < 0.03 in both settings;
  - every quoted count moves by < 10%;
  - the only rule failure is an unquoted small count (S3c − early mouse-only 13 → 10 when family
    pairs are dropped), caused by re-estimation, not by the dropped genes.
- **Statements to update in `docs/paper/draft.md`.**
  1. Line 145 ("a shared one-to-one ortholog space") and line 208 ("a one-to-one ortholog space"):
     say "reciprocal-best HCOP pairs". Methods (line 901) already says mutual-best.
  2. R3, mouse-only examples ("Fah, Comt, Maoa, Igfbp4 and Ebp"): Comt, Igfbp4 and Ebp are
     borderline. Each becomes indeterminate under one alternative pairing. Name Fah, Maoa and Hsd17b4
     as the robust examples, or add that the others are borderline.
  3. R3, the tyrosine-metabolism enrichment "only through three genes: Comt, Maoa and Fah": one of the
     three (Comt) is borderline, so the enrichment rests on two robust genes.
  4. R3, "Sterol synthesis and peroxisomal β-oxidation rest on single genes (Ebp, Hsd17b4)": Ebp is
     borderline.
  5. SM5: add one sentence: 1,051 of the 17,449 pairs lie in multi-member orthogroups; swapping in the
     main PT paralog or dropping them changes the index by ≤ 0.007 and quoted counts by < 10%
     (notebook 64).